## imports and a sample document

In [1]:
import re

document = """# Annual Report

This report covers the year in review.

## Medical Research

Our medical team studied a significant bug in the XDR-47 strain. The bug was not seen before. Researchers ran trials across three hospitals. Results were promising. A follow-up study begins next year.

## Software Engineering

Engineers fixed 247 bugs this year. The team shipped four major releases. Code review time fell by 30%. Test coverage rose to 85%. Next year focuses on performance.
"""

## size-based chunking (with overlap)

In [2]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0
    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])
        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text)
    return chunks

for i, c in enumerate(chunk_by_char(document), 1):
    print(f"--- Chunk {i} ---\n{c}\n")

--- Chunk 1 ---
# Annual Report

This report covers the year in review.

## Medical Research

Our medical team studied a significant bug in the XDR-47 strain. The bug

--- Chunk 2 ---
R-47 strain. The bug was not seen before. Researchers ran trials across three hospitals. Results were promising. A follow-up study begins next year.



--- Chunk 3 ---
 begins next year.

## Software Engineering

Engineers fixed 247 bugs this year. The team shipped four major releases. Code review time fell by 30%. T

--- Chunk 4 ---
 time fell by 30%. Test coverage rose to 85%. Next year focuses on performance.




Notice words cut in half at the edges, which is the weakness the lesson describes.

## structure-based chunking

In [3]:
def chunk_by_section(document_text):
    return re.split(r"\n## ", document_text)

for i, c in enumerate(chunk_by_section(document), 1):
    print(f"--- Section {i} ---\n{c}\n")

--- Section 1 ---
# Annual Report

This report covers the year in review.


--- Section 2 ---
Medical Research

Our medical team studied a significant bug in the XDR-47 strain. The bug was not seen before. Researchers ran trials across three hospitals. Results were promising. A follow-up study begins next year.


--- Section 3 ---
Software Engineering

Engineers fixed 247 bugs this year. The team shipped four major releases. Code review time fell by 30%. Test coverage rose to 85%. Next year focuses on performance.




Each chunk is a complete section, so “bugs” in software and medical research stay apart. The first chunk is the intro, and the others lose their ## marker because the split removes it.

## sentence-based chunking

In [4]:
def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    if overlap_sentences >= max_sentences_per_chunk:
        raise ValueError("overlap_sentences must be smaller than max_sentences_per_chunk")

    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    start_idx = 0
    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        chunks.append(" ".join(sentences[start_idx:end_idx]))
        if end_idx == len(sentences):
            break
        start_idx += max_sentences_per_chunk - overlap_sentences
    return chunks

for i, c in enumerate(chunk_by_sentence(document, 3, 1), 1):
    print(f"--- Chunk {i} ---\n{c}\n")

--- Chunk 1 ---
# Annual Report

This report covers the year in review. ## Medical Research

Our medical team studied a significant bug in the XDR-47 strain. The bug was not seen before.

--- Chunk 2 ---
The bug was not seen before. Researchers ran trials across three hospitals. Results were promising.

--- Chunk 3 ---
Results were promising. A follow-up study begins next year. ## Software Engineering

Engineers fixed 247 bugs this year.

--- Chunk 4 ---
## Software Engineering

Engineers fixed 247 bugs this year. The team shipped four major releases. Code review time fell by 30%.

--- Chunk 5 ---
Code review time fell by 30%. Test coverage rose to 85%. Next year focuses on performance.

--- Chunk 6 ---
Next year focuses on performance. 



I added ``if end_idx == len(sentences): break``, because the lesson’s version would otherwise emit a final chunk made only of overlap sentences.

## compare the three

In [5]:
for name, chunks in [
    ("size", chunk_by_char(document)),
    ("section", chunk_by_section(document)),
    ("sentence", chunk_by_sentence(document, 3, 1)),
]:
    print(name, "->", len(chunks), "chunks, longest:", max(len(c) for c in chunks), "chars")

size -> 4 chunks, longest: 150 chars
section -> 3 chunks, longest: 219 chars
sentence -> 6 chunks, longest: 170 chars
